<a href="https://colab.research.google.com/github/ga4gh/analytics-dashboard/blob/plenary-proof-of-concept/notebooks/Analytics_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analytics Dashboard

This notebook provides a proof of concept for a comprehensive analytics dashboard for exploring GA4GH initiatives.

### What This Notebook Does

This notebook automatically:
1. **Fetches**
     * GitHub repositories
     * PyPi libraries
     * Pubmed articles
2. **Analyzes the data** to show trends, statistics, and insights
3. **Creates interactive visualizations** to graphically present trends in the data

## How to Run This Notebook

1. **Click "Run All" in your Jupyter environment** - Everything will run automatically
2. **Wait for results** - The notebook will fetch data and create visualizations
3. **Scroll through the results** - Each section provides different insights

## What You Can Customize

### Advanced Changes
- Requires knowledge of Plotly and Dash libraries
- Can modify visualization types, styling, and data presented

## What You'll See

1. **Yearly Productivity** - A chart of repositories, articles and libraries created every year since the inception of GA4GH
   
## Important Notes

- **Processing time** - Initial run may take 2-3 minutes
- **Internet required** - Fetches data from API set up by the tech team
- **No data storage** - Results are temporary unless you save them

## Troubleshooting

**If visualizations don't appear:**
1. Ensure you have the required libraries installed
2. Try refreshing your browser
3. Check that JavaScript is enabled

**If results seem incomplete:**  
***Note: This is not a complete dataset. This is a POC and will be more complete in the future***
1. Try running individual sections instead of all at once


### Libraries Used in This Notebook

**dash** → Used to build interactive dashboard components such as graphs, layouts, and user inputs directly inside the notebook.

**requests** → Connects to the Analytics Dashboard DB which is curated by the Tech Team and retrieves data in JSON format.

**pandas** → Helps convert API responses into DataFrames for easy filtering, analysis, and visualization.

**json** → Used for handling and inspecting raw JSON responses.

**typing (List, Optional, Dict, Any)** → Adds optional type hints to improve code readability and maintainability.

**plotly.express** → Creates interactive charts (bar, line, pie, scatter) quickly and easily.

In [2]:
# Make sure Dash is installed
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 58.6 MB/s eta 0:00:00


In [17]:
# Some of the imports required to use the notebook
import requests
import json
import dash
from typing import List, Optional, Dict, Any
from dash import Dash, dcc, html, Input, Output, callback, dash_table, jupyter_dash
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from collections import Counter
from datetime import datetime


## Configurations & API Setup
---
**Explanation:**
- There are several endpoints for the Analytics Dashboard using the ```BASE_URL``` such as the ```TOTAL_PACKAGES_ENDPOINT``` as well as others listed in the cell below.
- ```get_repos(REPO_ENDPOINT, repos_list)```: This function will retrieve all the repos under the "ga4gh" org from the DB
  1. Makes a GET request to the specified endpoint.  
  2. Raises an error if the request fails.  
  3. Returns the response as JSON for further processing.

---

**Data Exploration:**  
- You can test this by running, for example:  
  ```python
  get_json(PM_KEYWORD)

In [4]:
# -----------------------------------------------------
# The setup section
# -----------------------------------------------------

# Base URL
BASE_URL = "http://analytics-staging.ga4gh.org:8000"

# Github Endpoints
REPO_ENDPOINT = f"{BASE_URL}/github/name/"
repos_list = ['ga4gh-schemas', 'ga4gh-server', 'compliance', 'dwg-website', 'gastore', 'mme-apis', 'metadata-team', 'g2p-team', 'benchmarking-tools', 'build', 'tool-registry-service-schemas', 'swg-ssig', 'tool-registry-server', 'workflow-execution-server', 'task-execution-server', 'task-execution-schemas', 'workflow-execution-service-schemas', 'ga4gh-consent-policy', 'hgvs-lib', 'tool-registry-validator', 'tool-registry-reference-implementation', 'vrs', 'hackathon2016', 'ga4gh-client', 'ga4gh-common', 'htsget', 'data-repository-service-schemas', 'ga4gh-converters', 'ADA-M', 'wiki', 'vrs-python', 'cloud-interop-testing', 'phenopacket-schema', 'large-scale-genomics-wiki', 'refget-compliance-suite', 'va-spec', 'variant-representation', 'approval-tracker', 'duri', 'refget-client', 'cloud-conformance-testing', 'data-object-service-schemas', 'w3id.org', 'data-security', 'pedigree', 'gh-openapi-docs', 'ga4gh-drs-client', 'refget-cloud', 'htsget-refserver', 'refget-loader', 'cloud-interop-ui', 'ga4gh-copyright-policy', 'TASC', 'standards-schema', 'htsget-compliance', 'ga4gh-registry', 'gatk', 'refget-compliance', 'htsjdk', 'ga4gh-bed', 'fasp-scripts', 'htsget-refserver-utils', 'ga4gh-ci', 'refget', 'reverse-lookup-spec', 'ga4gh-starter-kit-drs', 'vrs-vcf-alignment', 'pedigree-fhir-ig', 'vrsatile', 'ga4gh-starter-kit-docs', 'ga4gh-starter-kit-common', 'ga4gh-starter-kit-passport-broker', 'pedigree_family_history_terminology', 'ga4gh-starter-kit-wes', 'ga4gh-starter-kit', 'ga4gh-starter-kit-ui', 'machine-readable-consent-guidance', 'pedigree-tools', 'vrs-protobuf', 'ga4gh-starter-kit-utils', 'vrsatile-pydantic', 'ga4gh-testbed-lib', 'ga4gh-starter-kit-refget', 'ga4gh-testbed-ui', 'pedigree-validator', 'gks-metaschema', 'future-of-vcf', 'ga4gh-starter-kit-data-connect', 'cloud-best-practices', 'schema-registry-api', 'schema-registry-ui', 'schema-registry', 'tech-team', 'ga4gh-testbed-api', 'vrs-hackathons', 'Get-Started-with-GA4GH-APIs', 'ga4gh-starter-kit-passport-ui', 'vrs-phenopackets', 'cohort-rep-hackathon', 'product-process', 'openapi-test-runner', 'drs-compliance-suite', 'vrs-clojure', 'fasp-clients', 'sa-spec', 'gk-pilot', 'ga4gh-starter-kit-beacon', 'quality-control-wgs', 'Strategic-Refresh', 'drs-test-a-thon', 'experiments-metadata', 'ga4gh-pgx', 'compliance-tests-ga4gh-tes', 'ga4gh-crypt4gh', 'compliance-tests-ga4gh-wes', 'ga4gh-doi', 'gks-core', 'cat-vrs', 'compliance-tests-ga4gh-service-registry', 'ga4gh-testbed-api-aws-stack', 'gks2clinvar', 'human-pangenome-project', 'gks-validator', 'gks-portal', 'cat-vrs-python', 'va-spec-python', 'Website-flowcharts', 'gks-starter-repo']

# PyPi Endpoints
TOTAL_PACKAGES_ENDPOINT = f"{BASE_URL}/pypi/total-packages"
PACKAGE_VERSIONS_ENDPOINT = f"{BASE_URL}/pypi/package-versions"
RELEASES_OVER_YEARS_ENDPOINT = f"{BASE_URL}/pypi/releases-over-years"
ALL_SOURCES_COVERAGE_ENDPOINT = f"{BASE_URL}/pypi/all_sources_coverage"
PYPI_DETAILS_ENDPOINT = f"{BASE_URL}/pypi/project_details"

# PubMed Endpoints
PM_KEYWORD = f"{BASE_URL}/pubmed/articles/GA4GH"

# Function that will take a list of repos and retrieve them from the DB
def get_repos(endpoint: str, list_of_repos: List[str]) -> List[Dict[str, Any]]:

    items: List[Dict[str, Any]] = []
    url = endpoint

    for repo in list_of_repos:
        resp = requests.get(url + repo)
        resp.raise_for_status()
        items.append(resp.json()[0])

    return items

# Function to make a request to the endpoints to return data from the DB addressing any pagination issues
def get_json(endpoint: str, token: Optional[str] = None, per_page: int = 100) -> List[Dict[str, Any]]:

    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"

    params = {"per_page": per_page, "page": 1}
    items: List[Dict[str, Any]] = []
    url = endpoint

    while True:
        resp = requests.get(url, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if not isinstance(data, list):
            return data

        items.extend(data)

        if "next" in resp.links:
            url = resp.links["next"]["url"]
            params = None
        else:
            break

    return items

### Dynamic DataTable

This section creates a dynamic data table that displays the PyPi libraries relating to GA4GH products.  
The chart shows the **number of versions for each package** currently displayed in the table.

In [12]:
# -----------------------------------------------------
# 1. Total packages
# -----------------------------------------------------
total_packages_resp = get_json(TOTAL_PACKAGES_ENDPOINT)
total_packages = int(total_packages_resp.get("total_packages", 0))

# -----------------------------------------------------
# 2. Project details
# -----------------------------------------------------
details_data = get_json(PYPI_DETAILS_ENDPOINT)

# Build DataFrame
rows = []
for pkg in details_data:
    versions_list = pkg.get("versions", [])
    versions_count = versions_list[0].get("versions", 0) if versions_list else 0
    rows.append({
        "project_name": pkg.get("project_name"),
        "description": pkg.get("description"),
        "author_name": pkg.get("author_name"),
        "author_email": pkg.get("author_email"),
        "category": pkg.get("category"),
        "versions_count": versions_count,
    })

df = pd.DataFrame(rows)

# -----------------------------------------------------
# 3. Dynamic DataTable
# -----------------------------------------------------
from IPython.display import display, HTML
total_packages = len(df)
# Convert DataFrame to HTML with DataTables
html_str = df.to_html(classes="display", table_id="my_table", index=False)
html_template = f"""
<h3>Total PyPI Packages: {total_packages}</h3>
{html_str}
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.css">
<script type="text/javascript" charset="utf8" src="https://code.jquery.com/jquery-3.7.1.js"></script>
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.js"></script>
<script>
$(document).ready(function() {{
    $('#my_table').DataTable({{
        "pageLength": 10,
        "lengthMenu": [5, 10, 20, 50],
        "order": [[0, "asc"]],
        "searching": true,
        "info": true,
        "paging": true
    }});
}});
</script>
"""
# Display inline in notebook
display(HTML(html_template))


project_name,description,author_name,author_email,category,versions_count
data-repo-client,Data Repository API,OpenAPI Generator community,team@openapitools.org,Implementation,635
cwltool,Common workflow language reference implementation,Common workflow language working group,common-workflow-language@googlegroups.com,Implementation,437
pyphetools,Generate and work with GA4GH phenopackets,None,"Peter Robinson <peter.robinson@bih-charite.de>, Daniel Danis <daniel.gordon.danis@protonmail.com>",Implementation,199
planemo,Command-line utilities to assist in building tools for the Galaxy project (http://galaxyproject.org/).,None,Galaxy Project and Community <galaxy-committers@lists.galaxyproject.org>,Implementation,148
bento-lib,A set of common utilities and helpers for Bento platform services.,David Lougheed,david.lougheed@mail.mcgill.ca,Reference,105
bycon,A Python-based environment for the GA4GH Beacon genomics API,None,Michael Baudis <m@baud.is>,GA4GH Standard,73
sapporo,The sapporo-service is a standard implementation conforming to the Global Alliance for Genomics and Health (GA4GH) Workflow Execution Service (WES) API specification.,None,"""DDBJ (Bioinfomatics and DDBJ Center)"" <tazro.ohta@chiba-u.jp>",Implementation,58
variation-normalizer,VICC normalization routine for variations,"Alex Wagner, Kori Kuzma, James Stevenson",None,Implementation,56
ga4gh.vrs,GA4GH Variation Representation Specification (VRS) reference implementation,None,"Larry Babb <lbabb@broadinstitute.org>, Reece Hart <reecehart@gmail.com>, Andreas Prlic <andreas.prlic@gmail.com>, Alex Wagner <alex.wagner@nationwidechildrens.org>",GA4GH Standard,51
nmdc-runtime,A runtime system for NMDC data management and orchestration,None,None,Reference,44


### Bar Graph

This section creates a bar graph based on the data in the table above.

### Key Features:
- **Hover Tooltips** → Show extra information such as project name, category, and version count when hovering over the bar.  
- **Custom Styling** → Titles, colors, and hover formatting.

### Exploration Opportunities for Users:
- **Change the chart type** → Replace `"type": "bar"` with `"type": "scatter"` for a line chart.  
- **Adjust hover text** → Customize what appears on hover by editing the `hover_texts` list.  
- **Change colors** → Update `"marker": {"color": "#2E86C1"}` to another hex code (e.g., `"#E74C3C"` for red).
- **Modify axis titles** -> In `xaxis` and `yaxis`, replace `"project_name"` or `"versions_count"` with friendlier labels.

This visualization gives a **clear snapshot of version counts** for the subset of packages currently being inspected in the table.

In [9]:
# -----------------------------------------------------
# 4. Bar graph (Logic adapted from update_bar)
# -----------------------------------------------------
# Mimicking the "Page 1" logic from the original app (first 10 rows)
page_data = df.iloc[0:10]

# Create hover text: shorten long description
hover_texts = [
    f"<b>{row['project_name']}</b><br>"
    f"Category: {row.get('category', '')}<br>"
    f"Versions: {str(row.get('versions_count', ''))}"
    for _, row in page_data.iterrows()
]

fig_bar = go.Figure(data=[
    go.Bar(
        x=page_data["project_name"],      # X-axis: package names
        y=page_data["versions_count"],    # Y-axis: version counts
        marker={"color": "#2E86C1"},      # Bar color
        hovertext=hover_texts,            # Custom hover info
        hoverinfo="text"                  # Only show hover text
    )
])

fig_bar.update_layout(
    title={
        "text": "Package Versions Count (Top 10 Preview)",
        "x": 0.5,          # center title
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#2C3E50"}
    },
    xaxis={"title": "project_name", "tickangle": -45},
    yaxis={"title": "versions_count"},
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="#ffffff",
    margin={"b": 120}
)

### Pie Graph
This pie graph is based on the data table above displaying the categories of the PyPi libraries from implementations, references and standards.

In [11]:
# -----------------------------------------------------
# 5. Category distribution (Logic adapted from update_category_distribution)
# -----------------------------------------------------
# Count categories
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]

fig_pie = go.Figure(data=[
    go.Pie(
        labels=cat_counts["category"],
        values=cat_counts["count"],
        hole=0.4,
        textinfo="label+percent",
        hoverinfo="label+value+percent"
    )
])

fig_pie.update_layout(
    title={
        "text": "Category Distribution",
        "x": 0.5,          # center title
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#2C3E50"}
    },
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="#ffffff"
)


### Recent Update Dates for Github and PyPi

This line graph presents year-by-year update counts for PubMed articles versus GitHub repositories; hover over the points to view the updated repository and article totals as well as a list of articles and repositories.


In [14]:
# 1. Process GitHub Data
GA4GH_repos = get_repos(REPO_ENDPOINT, repos_list)
gh_names = [repo['name'] for repo in GA4GH_repos]
gh_dates = [repo['pushed_at'] for repo in GA4GH_repos]

df_gh = pd.DataFrame({
    "item": gh_names,
    "created_at": pd.to_datetime(gh_dates, utc=True)
})
df_gh["Source"] = "GitHub Repos"

# 2. Process PubMed Data
pubmed_releases = get_json(PM_KEYWORD)
pm_names = [release['title'] for release in pubmed_releases]
pm_dates = [release['publish_date'] for release in pubmed_releases]

df_pm = pd.DataFrame({
    "item": pm_names,
    "created_at": pd.to_datetime(pm_dates, utc=True, errors='coerce')
})
df_pm["Source"] = "PubMed Articles"

# 3. Combine GitHub and PubMed
df_raw = pd.concat([df_gh, df_pm])
df_raw = df_raw.dropna(subset=["created_at"])
df_raw["year"] = df_raw["created_at"].dt.year

# Group by Year and Source
final_df = df_raw.groupby(["year", "Source"]).agg({"item": list}).reset_index()
final_df["yearly_count"] = final_df["item"].apply(len)
final_df["items_str"] = final_df["item"].apply(lambda items: "<br>".join(items))

# Sort values
final_df = final_df.sort_values(["Source", "year"])

# 4. Create Graph
fig2 = px.line(
    final_df,
    x="year",
    y="yearly_count",
    color="Source",
    markers=True,
    title="Most Recent Update: GitHub & PubMed",
    labels={"year": "Year", "yearly_count": "Updates per Year"},
    custom_data=["items_str", "Source"]
)

# 5. Update Hover Template
fig2.update_traces(
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>" # Source Name
        "Year: %{x}<br>"
        "Updates: %{y}<br><br>"
        "<b>Updated Items:</b><br>%{customdata[0]}<extra></extra>"
    ),
    marker=dict(size=8),
)

fig2.update_layout(
    xaxis_title="Year",
    yaxis_title="Updates per Year",
    margin=dict(l=40, r=20, t=70, b=120),
    height=500,
    legend_title_text=""
)

### Top 8 Journals for GA4GH Articles

This graph displays the top 8 Journals in PubMed for both "GA4GH" and "Global Alliance for Genomics and Health" keywords.

In [18]:
def get_articles_data(keyword):
    """
    Fetches article data from the API for a given keyword.

    What it does:
    - Sends a request to our analytics dashboard API
    - Returns a list of articles if successful
    - Returns an empty list if there's an error
    - Prints error messages to help with troubleshooting

    Parameters:
    - keyword: The search term (e.g., "GA4GH")

    Returns:
    - List of article dictionaries with all the metadata

    DO NOT MODIFY: This function is used throughout the analytics
    """
    url = f"{BASE_URL}/pubmed/articles/{keyword}"
    response = requests.get(url)

    if response.status_code == 200:
        return response.json()
    print(f"Error fetching data for {keyword}: {response.status_code}")
    return []

keywords = [
    "GA4GH",
    "Global Alliance for Genomics and Health"
]

all_articles_data = {}

print("Fetching articles data for analytics")
print()

for keyword in keywords:
    print(f"Searching for: {keyword}")
    articles = get_articles_data(keyword)
    all_articles_data[keyword] = articles
    print(f"Found {len(articles)} articles")
    print()

print("Data collection complete")

def analyze_journals(articles, keyword, top_n=10):
    """
    Analyzes journal distribution for a specific keyword's articles.

    What it does:
    - Counts how many articles appear in each journal
    - Identifies the top publishing venues
    - Creates a horizontal bar chart visualization
    - Shows which journals are most active in this topic

    DO NOT MODIFY ANYTHING HERE
    """
    journals = [article.get("journal", "Unknown") for article in articles if article.get("journal")]

    if not journals:
        print(f"No journal data found for {keyword}")
        return

    journal_counts = Counter(journals)
    top_journals = journal_counts.most_common(top_n)

    # Create horizontal bar chart with Plotly
    journals_list, counts_list = zip(*top_journals, strict=False)

    # Truncate long journal names for display
    display_journals = [journal[:50] + "" if len(journal) > 50 else journal for journal in journals_list]

    fig = go.Figure(go.Bar(
        x=counts_list,
        y=display_journals,
        orientation='h',
        marker_color='lightcoral',
        text=counts_list,
        textposition='outside'
    ))

    fig.update_layout(
        title=f"Top {top_n} Journals for {keyword} Articles",
        xaxis_title="Number of Publications",
        yaxis_title="Journals",
        height=max(400, len(top_journals) * 40),
        margin=dict(l=200)  # Left margin for journal names
    )

    fig.show()

    print(f"\nTop {top_n} journals for {keyword}:")
    for i, (journal, count) in enumerate(top_journals, 1):
        print(f"  {i}. {journal}: {count} articles")

print("Creating Journal Analysis for all keywords")
print("This shows which journals publish articles for each keyword.")
print()

# ✨ CUSTOMIZABLE: Change this number to show more/fewer top journals
top_journals_to_show = 8  # Shows top 8 journals for each keyword

journals_analyzed = 0

# Analyze journals for each keyword in your list
for keyword in keywords:
    articles = all_articles_data.get(keyword, [])

    if articles:
        print(f"Analyzing journals for: {keyword}")
        print("-" * 60)

        analyze_journals(articles, keyword, top_n=top_journals_to_show)
        journals_analyzed += 1

        print("\n" + "="*80 + "\n")
    else:
        print(f"No articles found for {keyword} - skipping journal analysis")
        print()

print(f"\nJournal Analysis Complete")
print(f"Analyzed journals for {journals_analyzed} out of {len(keywords)} keywords")
if journals_analyzed > 0:
    print("Use this data to:")
    print("   • Identify the leading journals in your keyword")
    print("   • Track which journals are most active in different topics")
    print("   • Discover interdisciplinary publication patterns")

Fetching articles data for analytics
This may take a moment depending on how much data is available.

Searching for: GA4GH
Found 91 articles

Searching for: Global Alliance for Genomics and Health
Found 97 articles

Data collection complete
Creating Journal Analysis for all keywords
This shows which journals publish articles for each keyword.

Analyzing journals for: GA4GH
------------------------------------------------------------



Top 8 journals for GA4GH:
  1. Cell genomics: 13 articles
  2. Nucleic acids research: 8 articles
  3. Human mutation: 7 articles
  4. Bioinformatics (Oxford, England): 6 articles
  5. medRxiv : the preprint server for health sciences: 4 articles
  6. F1000Research: 4 articles
  7. NPJ genomic medicine: 3 articles
  8. Human genomics: 2 articles


Analyzing journals for: Global Alliance for Genomics and Health
------------------------------------------------------------



Top 8 journals for Global Alliance for Genomics and Health:
  1. Cell genomics: 9 articles
  2. Bioinformatics (Oxford, England): 7 articles
  3. NPJ genomic medicine: 4 articles
  4. medRxiv : the preprint server for health sciences: 4 articles
  5. F1000Research: 4 articles
  6. Human mutation: 4 articles
  7. Genetics in medicine : official journal of the American College of Medical Genetics: 3 articles
  8. Human genomics: 3 articles



Journal Analysis Complete
Analyzed journals for 2 out of 2 keywords
Use this data to:
   • Identify the leading journals in your keyword
   • Track which journals are most active in different topics
   • Discover interdisciplinary publication patterns
